# Assignment 3: Build a Link Predictor & Investigate Data Leakage

## 1. Overview and Problem Formulation
Link prediction aims to predict missing citation edges in the Cora dataset. In this assignment, we:
1. Treat the citation graph as **undirected** and train **without document category labels**.
2. Implement strict edge splitting (Train 85%, Val 5%, Test 10%) keeping both directions $(u, v)$ and $(v, u)$ in the same split.
3. Generate fixed, disjoint negative evaluation pairs excluding self-loops and observed edges.
4. Construct a **strictly leak-free message-passing graph** containing only training edges.
5. Implement **automated zero-leakage assertions**.
6. Compare a **Dot-Product Decoder** against a **Learned Symmetric MLP Decoder** ($s(u, v) = s(v, u)$) and a **Common Neighbors Baseline**.
7. Conduct a **deliberately leaky experiment** demonstrating why evaluating on test edges inside the message-passing graph is invalid.
8. Extract and discuss the top 10 high-confidence unobserved candidate links.

In [1]:
import os
import sys
sys.path.insert(0, r"D:\gnn-practical-assignments")

import torch
import numpy as np
import matplotlib.pyplot as plt
from assignment3_link_prediction.split_utils import create_link_prediction_splits, verify_zero_leakage
from assignment3_link_prediction.models_link import test_decoder_symmetry
from assignment3_link_prediction.train_link_predictor import run_assignment3_experiments
%matplotlib inline
print("Assignment 3 environment loaded.")

C:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Assignment 3 environment loaded.


## 2. Automated Zero-Leakage Verification
We split edges into train/val/test and execute automated assertions proving that no held-out edges exist in the message-passing graph.

In [2]:
splits = create_link_prediction_splits(seed=42)
verify_zero_leakage(splits)

[SUCCESS] Zero-leakage verified: 0 held-out edges exist in the message-passing graph.
[SUCCESS] Zero-leakage verified: 0 held-out edges exist in the message-passing graph.


## 3. Decoder Mathematical Symmetry Proof
We test our Symmetric MLP Decoder to ensure $s(u, v) = s(v, u)$ for undirected edge pairs.

In [3]:
test_decoder_symmetry()

MLP Decoder Score (u, v): 0.45939454
MLP Decoder Score (v, u): 0.45939454
[SUCCESS] MLP Decoder produces identical scores when node pair is reversed.


## 4. Benchmarks, Leaky Comparison & Candidate Discovery
We train the models, compare against the Common Neighbors heuristic, demonstrate the artificial jump from data leakage, and extract the top 10 candidate citations.

In [4]:
run_assignment3_experiments()

ASSIGNMENT 3: LINK PREDICTION BENCHMARKS & LEAKAGE INVESTIGATION


[SUCCESS] Zero-leakage verified: 0 held-out edges exist in the message-passing graph.


[SUCCESS] Zero-leakage verified: 0 held-out edges exist in the message-passing graph.


[SUCCESS] Zero-leakage verified: 0 held-out edges exist in the message-passing graph.


[SUCCESS] Zero-leakage verified: 0 held-out edges exist in the message-passing graph.



RESULTS SUMMARY TABLE:
Model / Approach                         | Test ROC-AUC    | Test Average Precision (AP)
-------------------------------------------------------------------------------------
1. Common Neighbors (Baseline Heuristic) | 0.7211          | 0.7181                   
2. GCN + Dot-Product Decoder (Valid)     | 0.7405          | 0.7441                   
3. GCN + Symmetric MLP Decoder (Valid)   | 0.6957          | 0.7225                   
4. GCN + Dot-Product (DELIBERATELY LEAKY) | 0.8142          | 0.8135                   

WHY THE LEAKY EXPERIMENT IS INVALID:
Notice that the leaky model scores 81.42% ROC-AUC vs 74.05% for the valid model.
This +7.38% jump is completely artificial! Because test edges were left in the message-passing graph,
the GCN directly passed node features across the test edges before the decoder was even queried.
In production, future links do not exist yet; using leaky metrics results in catastrophic failure in real-world deployment.

TOP 10 HI